### Load Data
*Loading the raw file and working on a copy, so the original is never changed.*


In [2]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/raw/ecommerce_orders_raw.csv")
df = raw.copy()
print("Start:", df.shape)

Start: (5020, 28)


**Result:** 5,020 rows and 28 columns, same as in Step 1.

### Remove Extra Spaces
*Trimming leading and trailing spaces in all text columns first, so that values such as ` m ` and `Germany ` match their clean versions.*

In [3]:
for col in df.select_dtypes(exclude="number").columns:
    df[col] = df[col].str.strip()

**Result:** Spaces removed from all text columns. In Step 1, `Gender` (820 rows), `Country` (458 rows) and `Customer_Name` (8 rows) were affected.

### Verifying that no spaces remain.


In [4]:
for c in df.select_dtypes(exclude="number").columns:
    assert df[c].dropna().str.contains(r"^\s|\s$").sum() == 0

   **Check:** After trimming, no text column has leading or trailing spaces (the assertion passed).

### Convert Order_Date
*`Order_Date` has 3 text formats. Each format is parsed separately and the results are combined. The slash format is day/month/year (confirmed in Step 1: the first part is sometimes greater than 12, the second never).*

In [5]:
# errors="coerce" turns a value that does not match the format into NaT instead of raising an error
# fillna chains the three formats: a row missed by one format is picked up by the next

iso = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(df["Order_Date"], format="%d/%m/%Y", errors="coerce")
mon = pd.to_datetime(df["Order_Date"], format="%b %d, %Y", errors="coerce")
df["Order_Date"] = iso.fillna(dmy).fillna(mon)

print("Failed:", df["Order_Date"].isna().sum())
print(df["Order_Date"].min(), df["Order_Date"].max())

Failed: 0
2023-01-01 00:00:00 2025-12-31 00:00:00


**Result:** All dates converted (0 failed). The data covers 2023-01-01 to 2025-12-31.

### Standardize Gender, Country and Category
*Each column gets a mapping dictionary: lowercase value -> standard name. Values not in the mapping become missing, so the result is checked afterwards.*

In [6]:
# .map() turns any value that is not in the dictionary into NaN, so check the counts afterwards

gender_map = {
    "m": "Male", "male": "Male",
    "f": "Female", "female": "Female", "woman": "Female",
}

country_map = {
    "de": "Germany", "germany": "Germany", "deutschland": "Germany",
    "fr": "France", "france": "France",
    "at": "Austria", "austria": "Austria", "österreich": "Austria",
    "nl": "Netherlands", "netherlands": "Netherlands", "the netherlands": "Netherlands",
    "be": "Belgium", "belgium": "Belgium",
    "ch": "Switzerland", "switzerland": "Switzerland", "schweiz": "Switzerland",
}

category_map = {
    "electronic": "Electronics", "electronics": "Electronics", "elec": "Electronics",
    "office": "Office",
    "home & kitchen": "Home & Kitchen", "home and kitchen": "Home & Kitchen", "home_kitchen": "Home & Kitchen",
    "fashion": "Fashion",
    "sports": "Sports", "sport": "Sports",
}

# .map() turns any value that is not in the dictionary into NaN, so the counts are checked afterwards
df["Gender"] = df["Gender"].str.lower().map(gender_map)
df["Country"] = df["Country"].str.lower().map(country_map)
df["Category"] = df["Category"].str.lower().map(category_map)

In [7]:
print(df["Gender"].value_counts(dropna=False), "\n")
print(df["Country"].value_counts(dropna=False), "\n")
print(df["Category"].value_counts(dropna=False))

Gender
Male      2424
Female    2422
NaN        174
Name: count, dtype: int64 

Country
Germany        2190
France          774
Austria         594
Netherlands     594
Switzerland     477
Belgium         391
Name: count, dtype: int64 

Category
Electronics       1532
Office            1021
Home & Kitchen    1020
Fashion            737
Sports             710
Name: count, dtype: int64


**Result:**
- Gender: 12 spellings -> 2 (Male, Female). Missing count is unchanged at 174.
- Country: 23 labels -> 6 countries, no new missing values.
- Category: 17 labels -> 5 categories, no new missing values.

### Remove Duplicates
*In Step 1, the number of exact duplicate rows equaled the number of repeated `Order_ID` values, so every repeat is a full copy and can be removed safely.*

In [8]:
before = len(df)
df = df.drop_duplicates()
print("Removed:", before - len(df))
print("Order_ID unique?", df["Order_ID"].is_unique)

Removed: 20
Order_ID unique? True


**Result:** Removed 20 rows (0.4%). `Order_ID` is now unique. The dataset now has 5,000 rows.

### Impossible Values
*Setting impossible values to missing (they cannot be trusted), and recovering `Quantity` from the `Profit` formula where possible.*

In [9]:
print("Age outside 16-90:", (~df["Age"].dropna().between(16, 90)).sum())
print("Discount outside 0-0.9:", (~df["Discount"].dropna().between(0, 0.9)).sum())
print("Rating outside 1-5:", (~df["Customer_Rating"].dropna().between(1, 5)).sum())
print("Delivery_Days < 0:", (df["Delivery_Days"] < 0).sum())

Age outside 16-90: 4
Discount outside 0-0.9: 2
Rating outside 1-5: 2
Delivery_Days < 0: 1


In [10]:
df.loc[~df["Age"].between(16, 90), "Age"] = np.nan
df.loc[~df["Discount"].between(0, 0.9), "Discount"] = np.nan
df.loc[~df["Customer_Rating"].between(1, 5), "Customer_Rating"] = np.nan
df.loc[df["Delivery_Days"] < 0, "Delivery_Days"] = np.nan

In [11]:
bad_qty = df["Quantity"] <= 0
recovered = ((df["Sales"] - df["Profit"]) / df["Cost"]).round()
df.loc[bad_qty, "Quantity"] = recovered[bad_qty].astype("int64")
print(df.loc[bad_qty, ["Quantity", "Unit_Price", "Discount", "Sales"]])

      Quantity  Unit_Price  Discount  Sales
1188         1       24.63     0.081  22.64
2929         2       34.68     0.111  61.68
4796         2       73.70       NaN  95.81


In [12]:
assert df["Quantity"].between(1, 5).all()
assert df["Age"].dropna().between(16, 90).all()
assert df["Discount"].dropna().between(0, 0.9).all()
assert df["Customer_Rating"].dropna().between(1, 5).all()
assert (df["Delivery_Days"].dropna() >= 0).all()

**Result:** Set to missing: Age (4 values), Discount (2), Customer_Rating (2), Delivery_Days (1). Quantity was recovered for 3 rows (0, -2, -1 -> 1, 2, 2) using `Quantity = (Sales - Profit) / Cost`.

### Missing Values
*Each column is handled differently: Discount is recovered from the Sales formula, City is partly recovered from Region, descriptive columns get "Unknown", and numeric columns (Age, Delivery_Days, Customer_Rating) stay missing.*

In [13]:
missing = df.isna().sum().to_frame("missing")
missing[missing["missing"] > 0].sort_values("missing", ascending=False)

,missing
Customer_Rating,285
Delivery_Days,202
Discount,199
Gender,174
Age,171
Payment_Method,135
Marketing_Channel,130
Customer_Segment,128
City,128
Customer_Name,121


**Note:** The counts differ slightly from Step 1. The table shows why.

| Column | Step 1 | Duplicates removed | Impossible values set to missing | Now |
|--------|--------|--------------------|----------------------------------|-----|
| Age | 168 | -1 | +4 | 171 |
| Customer_Rating | 283 | 0 | +2 | 285 |
| Delivery_Days | 202 | -1 | +1 | 202 |
| Discount | 199 | -2 | +2 | 199 |
| Customer_Name | 122 | -1 | 0 | 121 |
| Marketing_Channel | 132 | -2 | 0 | 130 |

The other columns (Gender, City, Customer_Segment, Payment_Method) are unchanged.

In [14]:
known = df[df["Discount"].notna()]
estimated = 1 - known["Sales"] / (known["Quantity"] * known["Unit_Price"])
error = (estimated - known["Discount"]).abs()
print("Largest error:", error.max().round(5))

Largest error: 0.00083


In [15]:
missing_disc = df["Discount"].isna()
df.loc[missing_disc, "Discount"] = (
    1 - df["Sales"] / (df["Quantity"] * df["Unit_Price"])
).round(3)[missing_disc]
print("Discount missing after:", df["Discount"].isna().sum())

Discount missing after: 0


In [16]:
unknown_cols = ["Customer_Name", "Gender", "Customer_Segment",
                "Payment_Method", "Marketing_Channel", "City"]
for c in unknown_cols:
    df[c] = df[c].fillna("Unknown")

missing = df.isna().sum().to_frame("missing")
missing[missing["missing"] > 0]

,missing
Age,171
Delivery_Days,202
Customer_Rating,285


In [17]:
assert df["Discount"].notna().all()
assert df[unknown_cols].notna().all().all()

**Result:**
- **Discount:** all 199 missing values recovered from the Sales formula.
- **City:** 70 recovered from Region, 58 set to "Unknown".
- **Filled with "Unknown":** Gender (174), Payment_Method (135), Marketing_Channel (130), Customer_Segment (128) and Customer_Name (121).
- **Left missing on purpose:** Age (171), Delivery_Days (202) and Customer_Rating (285). Filling numbers with an average would add made-up values, and averages ignore missing values anyway.

### Logical Consistency: Order Status
*Cancelled orders were never delivered, so they should not have delivery days or ratings. `Returned` is checked against `Order_Status`.*

In [18]:
print(df["Order_Status"].value_counts(), "\n")
for status in ["Cancelled", "Returned", "Delivered"]:
    part = df[df["Order_Status"] == status]
    print(status, len(part),
          "| with rating:", part["Customer_Rating"].notna().sum(),
          "| with delivery days:", part["Delivery_Days"].notna().sum())
print("\nReturned matches Order_Status:",
      (df["Returned"] == (df["Order_Status"] == "Returned").astype(int)).all())

Order_Status
Delivered    4400
Returned      344
Cancelled     256
Name: count, dtype: int64 

Cancelled 256 | with rating: 237 | with delivery days: 247
Returned 344 | with rating: 318 | with delivery days: 329
Delivered 4400 | with rating: 4160 | with delivery days: 4222

Returned matches Order_Status: True


In [19]:
cancelled = df["Order_Status"] == "Cancelled"
print("Setting to missing -> rating:", df.loc[cancelled, "Customer_Rating"].notna().sum(),
      "| delivery days:", df.loc[cancelled, "Delivery_Days"].notna().sum())
df.loc[cancelled, ["Customer_Rating", "Delivery_Days"]] = np.nan

Setting to missing -> rating: 237 | delivery days: 247


In [20]:
assert (df["Returned"] == (df["Order_Status"] == "Returned").astype(int)).all()

In [21]:
assert df.loc[cancelled, ["Customer_Rating", "Delivery_Days"]].isna().all().all()
print(df["Customer_Rating"].isna().sum(), df["Delivery_Days"].isna().sum())

522 449


**Result:**
- Rating and delivery days set to missing for the 256 cancelled orders (237 ratings and 247 delivery days removed).
- `Returned` is identical to `Order_Status = Returned` (344 rows); both columns are kept and checked with an assertion.
- This changes the averages very little (mean rating 3.974 -> 3.971, mean delivery days 4.349 -> 4.348); the change is about correct logic, not about the numbers.